In [1]:
%display latex

### Intro

Sec. 3.1 of https://arxiv.org/html/2503.06780v2#S3 reorganizes the Picard–Fuchs data by noting that the action and its derivatives span a finite vector space, then encodes the PF equations as a Gauss–Manin connection matrix one-form in the
derivative frame basis.  The period derivatives are mapped to column vectors, and the PF equations are encoded in a connection matrix whose columns are read from the PF equations.

Calculations in the tilted case start one level earlier: we choose explicit residue/cohomology representatives like
$\omega_1 = \Omega/f$ and $\omega_2 = z^4 \Omega/f^2$ where $f=0$ is the weighted projective variety
$$ f  = 1/2 y^2 + x^4 - \mu_2x^3z + (E-2)x^2z^2 + (\mu_1+\mu_2)xz^3 + (\mu_0-E+1)z^4. $$
From this we have $ \partial_E f = z^2(x^2-z^2)$ so
$\partial_E (\Omega/f) = -(\Omega\partial_E f)f^{-2} = -z^2(x^2-z^2)\Omega f^{-2} $.

The pain point is that $\partial_E \Pi$ is not simply the period of $z^4 \Omega/f^2$. It is the class
 $-z^2(x^2-z^2)\Omega f^{-2}$ and only after one GD reduction does this become
 $\partial_E\omega_1 = a(E,\mu)\omega_1 + b(E,\mu)\omega_2$ in cohomology. So the derivative-frame basis is $\epsilon_1=\omega_1$ and
 $\epsilon_2 = \partial_E\omega_1 =a(E,\mu)\omega_1 + b(E,\mu)\omega_2$ Or
 $$\begin{pmatrix} \epsilon_1 \\ \epsilon_2 \end{pmatrix} = P \begin{pmatrix} \omega_1 \\ \omega_2 \end{pmatrix}$$
 where
 $$ P= \begin{pmatrix}  1 & 0 \\ a & b \end{pmatrix}$$

Then the GM matrices transform by the  gauge/frame-change formula
 $$ A^{(\epsilon)}_\lambda = (\partial_\lambda P +  P A_{\lambda}^{(\omega)})P^{-1} $$


 The important thing is that $A^{(\epsilon)}_\lambda$ is what should match the derivative-frame/PF matrix.

In [2]:
F0 = PolynomialRing(QQ, 'mu0,mu1,mu2,E')
F0.inject_variables()
mu0,mu1,mu2,E = F0.gens()
K = F0.fraction_field()

# weighted projective space. reverse degree lex ordering with 1,2,1 weighting of x,y,z
S.<x,y,z> = PolynomialRing(K, order=TermOrder('wdegrevlex', (1,2,1)))

# quartic projective variety
f  = 1/2*y^2 + x^4 - mu2*x^3*z + (E-2)*x^2*z^2 + (mu1+mu2)*x*z^3 + (mu0-E+1)*z^4
fx = diff(f,x)
fy = diff(f,y)
fz = diff(f,z)
fE = x^2*z^2-z^4

# Jac ideal
J = S.ideal([fx,fy,fz])
GB = J.groebner_basis()

Defining mu0, mu1, mu2, E


In [3]:
# Griffiths-Dwork pole reduction
# This matches Max's earlier "Reduction of Pole order" section

def NormalFormGD(p, J):
    r = p.reduce(J)
    qs = (p - r).lift(J)
    return r, qs   # remainder and quo

def PoleReduceGD(p, J, degf=4, weights=(1,2,1)):
    r, qs = NormalFormGD(p, J)
    xs = J.ring().gens()
    wdim = sum(weights)

    divq = sum(qs[i].derivative(xs[i]) for i in range(len(xs)))
    hcs = divq.homogeneous_components()

    out = r.parent()(0)
    for h in sorted(hcs):
        # h is the weighted degree after lowering the pole order.
        # If h = m_new deg(f) - sum(weights), divide by m_new.
        m_new = QQ((h + wdim)/degf)
        out += hcs[h]/m_new

    return out + r

#  completely reduce the polynomial
def PoleReduceFullGD(p, J, degf=4, weights=(1,2,1)):
    wdim = sum(weights)
    hmax = max(p.homogeneous_components())
    maxPoleOrder = ZZ((hmax + wdim)/degf)

    out = p
    for _ in range(maxPoleOrder - 1):
        out = PoleReduceGD(out, J, degf=degf, weights=weights)

    return out.reduce(J)

In [4]:
# parametric derivs. in the "new" coho. basis Ω/f, z^4 Ω/f^2
fE   = x^2*z^2 - z^4
fmu0 = z^4
fmu1 = x*z^3
fmu2 = x*z^3 - x^3*z

# First rows: ∂_λ ω0
r1_E   = PoleReduceFullGD(-fE,   J)
r1_mu0 = PoleReduceFullGD(-fmu0, J)
r1_mu1 = PoleReduceFullGD(-fmu1, J)
r1_mu2 = PoleReduceFullGD(-fmu2, J)

# Second rows: ∂_λ ω1
r2_E   = PoleReduceFullGD(-2*z^4*fE,   J)
r2_mu0 = PoleReduceFullGD(-2*z^4*fmu0, J)
r2_mu1 = PoleReduceFullGD(-2*z^4*fmu1, J)
r2_mu2 = PoleReduceFullGD(-2*z^4*fmu2, J)


In [5]:
def GM_entries(r1, r2):
    a11 = r1.constant_coefficient()
    a12 = r1.monomial_coefficient(z^4)

    a21  = r2.constant_coefficient()
    a22 = r2.monomial_coefficient(z^4)

    return a11, a12, a21, a22

a11_E, a12_E, a21_E, a22_E = GM_entries(r1_E, r2_E)
a11_mu0, a12_mu0, a21_mu0, a22_mu0 = GM_entries(r1_mu0, r2_mu0)
a11_mu1, a12_mu1, a21_mu1, a22_mu1 = GM_entries(r1_mu1, r2_mu1)
a11_mu2, a12_mu2, a21_mu2, a22_mu2 = GM_entries(r1_mu2, r2_mu2)

A_E = matrix([[a11_E, a12_E], [a21_E, a22_E]])
A_mu0 = matrix([[a11_mu0, a12_mu0], [a21_mu0, a22_mu0]])
A_mu1 = matrix([[a11_mu1, a12_mu1], [a21_mu1, a22_mu1]])
A_mu2 = matrix([[a11_mu2, a12_mu2], [a21_mu2, a22_mu2]])

print("A_E =")
show(A_E)

print("A_mu0 =")
show(A_mu0)

print("A_mu1 =")
show(A_mu1)

A_E =


[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

A_mu0 =


[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

A_mu1 =


[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

Check that $F[A]=0$ where $$F[A]=dA + A \wedge A $$

In [6]:
flat_Emu0 = (
        A_mu0.apply_map(lambda x: x.derivative(E))
        - A_E.apply_map(lambda x: x.derivative(mu0))
        + A_mu0*A_E
        - A_E*A_mu0
)

flat_Emu0 = flat_Emu0.apply_map(lambda x: x if x == 0 else x.factor())

print("Flatness check:")
show(flat_Emu0)

Flatness check:


[0 0]
[0 0]

In [7]:
flat_mu0mu1 = (
        A_mu0.apply_map(lambda x: x.derivative(mu1))
        - A_mu1.apply_map(lambda x: x.derivative(mu0))
        + A_mu0*A_mu1
        - A_mu1*A_mu0
)

flat_mu0mu1 = flat_mu0mu1.apply_map(lambda x: x if x == 0 else x.factor())

print("Flatness check:")
show(flat_mu0mu1)

Flatness check:


[0 0]
[0 0]

In [8]:
def symmetric_limit(p):
    R = p.parent()
    K = R.base_ring()
    mu0K, mu1K, mu2K, EK = K.gens()
    return p.map_coefficients(lambda c: c.subs({mu1K: K(0), mu2K: K(0)}))


# Symmetric-limit reduced numerators
r1_E_sym = symmetric_limit(r1_E)
r2_E_sym = symmetric_limit(r2_E)

r1_mu0_sym = symmetric_limit(r1_mu0)
r2_mu0_sym = symmetric_limit(r2_mu0)

r1_mu1_sym = symmetric_limit(r1_mu1)
r2_mu1_sym = symmetric_limit(r2_mu1)

In [9]:
def GM_entries_from_reduced(r1, r2):
    # extract the leading terms
    a11 = r1.constant_coefficient()
    a12 = r1.monomial_coefficient(z^4)

    a21 = r2.constant_coefficient()
    a22 = r2.monomial_coefficient(z^4)

    return a11, a12, a21, a22


# symmetric limit (tag quantities  with an s)
a11_E_s, a12_E_s, a21_E_s, a22_E_s = GM_entries_from_reduced(r1_E_sym, r2_E_sym)
a11_mu0_s, a12_mu0_s, a21_mu0_s, a22_mu0_s = GM_entries_from_reduced(r1_mu0_sym, r2_mu0_sym)

# rebuild the matrix
A_E_sym = matrix([[a11_E_s, a12_E_s], [a21_E_s, a22_E_s]])
A_mu0_sym = matrix([[a11_mu0_s, a12_mu0_s], [a21_mu0_s, a22_mu0_s]])

print("A_E_sym =")
show(A_E_sym)

print("A_mu0_sym =")
show(A_mu0_sym)

A_E_sym =


[                                                                                                                                            (-1)/(2*E - 4)                                                                                                                                        (2*mu0 - E)/(E - 2)]
[                                              (-6*mu0 + 3*E)/(-4*mu0*E^3 + 4*E^4 + 16*mu0^2*E - 8*mu0*E^2 - 12*E^3 - 32*mu0^2 + 48*mu0*E + 8*E^2 - 32*mu0) (mu0*E^2 - 3*E^3 + 12*mu0^2 - 12*mu0*E + 13*E^2 - 4*mu0 - 8*E)/(-2*mu0*E^3 + 2*E^4 + 8*mu0^2*E - 4*mu0*E^2 - 6*E^3 - 16*mu0^2 + 24*mu0*E + 4*E^2 - 16*mu0)]

A_mu0_sym =


[                                                                         0                                                                         -1]
[             3/(-4*mu0*E^2 + 4*E^3 + 16*mu0^2 - 16*mu0*E - 4*E^2 + 16*mu0) (E^2 - 8*mu0 + 4*E - 4)/(-mu0*E^2 + E^3 + 4*mu0^2 - 4*mu0*E - E^2 + 4*mu0)]

In [10]:
show(factor(denominator(A_E_sym[1][1].subs(E=1 - E, mu0=-mu0))))

(2) * (-mu0 + E) * (E + 1) * (E^2 + 4*mu0 - 2*E + 1)

In [11]:
show(factor(numerator(A_E_sym[1][1].subs(E=1-E,mu0=-mu0))))

-mu0*E^2 + 3*E^3 + 12*mu0^2 - 10*mu0*E + 4*E^2 + 15*mu0 - 9*E + 2

In [12]:
for name, val in [
    ("a11_E_s", a11_E_s), ("a12_E_s", a12_E_s),
    ("a21_E_s", a21_E_s), ("a22_E_s", a22_E_s),
    ("a11_mu0_s", a11_mu0_s), ("a12_mu0_s", a12_mu0_s),
    ("a21_mu0_s", a21_mu0_s), ("a22_mu0_s", a22_mu0_s)
]:
    if val != 0:
        print(name, "den =", factor(val.denominator()))

a11_E_s den = (2) * (E - 2)
a12_E_s den = E - 2
a21_E_s den = (4) * (-mu0 + E - 1) * (E - 2) * (E^2 - 4*mu0)
a22_E_s den = (2) * (-mu0 + E - 1) * (E - 2) * (E^2 - 4*mu0)
a12_mu0_s den = 1
a21_mu0_s den = (4) * (-mu0 + E - 1) * (E^2 - 4*mu0)
a22_mu0_s den = (-mu0 + E - 1) * (E^2 - 4*mu0)


Check flatness $F[A]=0$ of the symmetric s=0 reduction

In [13]:
flat_sym = (
        A_mu0_sym.apply_map(lambda x: x.derivative(E))
        - A_E_sym.apply_map(lambda x: x.derivative(mu0))
        + A_mu0_sym*A_E_sym
        - A_E_sym*A_mu0_sym
)

flat_sym = flat_sym.apply_map(lambda x: x if x == 0 else x.factor())

print("Flatness check:")
show(flat_sym)

Flatness check:


[0 0]
[0 0]

In [14]:
# Suppose the corrected symmetric GM matrices are
# A_E_sym   = matrix([[dE, cE], [betaE, alphaE]])
# A_mu0_sym = matrix([[dM, cM], [betaM, alphaM]])

dE, cE = A_E_sym[0,0], A_E_sym[0,1]
bE, aE = A_E_sym[1,0], A_E_sym[1,1]

dM, cM = A_mu0_sym[0,0], A_mu0_sym[0,1]
bM, aM = A_mu0_sym[1,0], A_mu0_sym[1,1]

In [15]:
# Define variables for coefficients in derivative basis
def express_in_derivative_basis(row_period, S):
    """
    Given a row vector row_period = [A, B] representing A*N0 + B*N1,
    solve for coeffs [v0,vE,vM] such that

        row_period = [v0,vE,vM] * S

    where S maps period basis to derivative basis.
    """
    K = row_period[0].parent()
    v0, vE, vM = var('v0 vE vM')

    # Work symbolically for solving
    eq0 = symbolic_expression(row_period[0]) == v0*symbolic_expression(S[0,0]) + vE*symbolic_expression(S[1,0]) + vM*symbolic_expression(S[2,0])
    eq1 = symbolic_expression(row_period[1]) == v0*symbolic_expression(S[0,1]) + vE*symbolic_expression(S[1,1]) + vM*symbolic_expression(S[2,1])

    sol = solve([eq0, eq1], [v0, vE, vM], solution_dict=True)

    # There is generally one-parameter freedom because derivative basis is redundant.
    # Choose vM=0 if free, or use the first solution Sage gives.
    return sol

In [16]:
def express_with_v0_zero(row_period, S):
    K = row_period[0].parent()
    vE, vM = var('vE vM')

    eq0 = symbolic_expression(row_period[0]) == vE*symbolic_expression(S[1,0]) + vM*symbolic_expression(S[2,0])
    eq1 = symbolic_expression(row_period[1]) == vE*symbolic_expression(S[1,1]) + vM*symbolic_expression(S[2,1])

    sol = solve([eq0, eq1], [vE, vM], solution_dict=True)[0]

    return [K(0), K(sol[vE]), K(sol[vM])]

In [24]:
def simp(x):
    x = parent(x)(x) if hasattr(x, "parent") else x
    if x == 0:
        return x
    try:
        return x.factor()
    except Exception:
        try:
            return symbolic_expression(x).simplify_rational()
        except Exception:
            return x

def deriv(x, var):
    return x.derivative(var)


def build_U_matrices(AE, AM, Evar, Mvar):
    K = AE.base_ring()

    dE, cE = AE[0,0], AE[0,1]
    bE, aE = AE[1,0], AE[1,1]

    dM, cM = AM[0,0], AM[0,1]
    bM, aM = AM[1,0], AM[1,1]

    def deriv(x, var):
        return K(x.derivative(var))

    def simp(x):
        x = K(x)
        if x == 0:
            return x
        try:
            return K(x.factor())
        except Exception:
            return x

    def to_deriv_basis(A, B):
        A = K(A)
        B = K(B)

    # derivative frame:
    # N_E = dE N0 + bE N1
    # N_M = dM N0 + bM N1
    #
    # Gauge choice: no N_E term.
        uE = K(0)
        #uM = K(B / cM)
        #u0 = K(A - B*dM/cM)
        uM = K(B / bM)
        u0 = K(A - B*dM/bM)

        return [simp(u0), simp(uE), simp(uM)]

    # E derivatives
    A_EE = deriv(dE,Evar) + dE*dE + cE*bE
    B_EE = deriv(cE,Evar) + dE*cE + cE*aE

    A_EM = deriv(dM,Evar) + dM*dE + cM*bE
    B_EM = deriv(cM,Evar) + dM*cE + cM*aE

    row0_UE = [K(0), K(1), K(0)]
    row1_UE = to_deriv_basis(A_EE, B_EE)
    row2_UE = to_deriv_basis(A_EM, B_EM)

    UE = matrix(K, [row0_UE, row1_UE, row2_UE])

    # mu derivatives
    A_ME = deriv(dE,Mvar) + dE*dM + cE*bM
    B_ME = deriv(cE,Mvar) + dE*cM + cE*aM

    A_MM = deriv(dM,Mvar) + dM*dM + cM*bM
    B_MM = deriv(cM,Mvar) + dM*cM + cM*aM

    row0_UM = [K(0), K(0), K(1)]
    row1_UM = to_deriv_basis(A_ME, B_ME)
    row2_UM = to_deriv_basis(A_MM, B_MM)

    UM = matrix(K, [row0_UM, row1_UM, row2_UM])

    return UE.apply_map(simp), UM.apply_map(simp)

In [25]:
def period_row_to_derivative_row(A, B, dE, bE, dM, bM):
    K = A.parent()
    uE = K(0)
    uM = B / bM
    u0 = A - B*dM/bM
    #uM = B / cM
    #u0 = A - B*dM/cM
    return [simp(u0), uE, simp(uM)]

In [37]:
A1 = (dP + P * A_E_sym) * P.inverse()
A2 = P.inverse() * A_E_sym * P + P.inverse() * dP

simp(A1[0,1]), simp(A2[0,1])

(1, (E - 2)^-2 * (2*mu0 - E)^2)

In [39]:
A_E_deriv = simp(A1)

$$A^{\rm deriv}_E = \begin{pmatrix} 0 & 1 \\ \alpha & \beta \end{pmatrix}$$
should give the PF eqns $\partial^2 N = \alpha N_0 + \beta \partial_E N_0 $

In [42]:
A_E_deriv

[                                                                                                                                   0                                                                                                                                    1]
[                    -1/4*(E^2 - (5*E - 4)*mu0 + 2*mu0^2)/(E^4 - E^3 + 2*(E^2 + 6*E - 4)*mu0^2 - 8*mu0^3 - (3*E^3 + 2*E^2 - 4*E)*mu0) -(2*E^3 + 4*(E + 1)*mu0^2 - E^2 - (7*E^2 - 4*E + 4)*mu0)/(E^4 - E^3 + 2*(E^2 + 6*E - 4)*mu0^2 - 8*mu0^3 - (3*E^3 + 2*E^2 - 4*E)*mu0)]

In [27]:
K = A_E_sym[0,0].parent()
mu0K, mu1K, mu2K, EK = K.gens()

U_E_sym, U_mu0_sym = build_U_matrices(A_E_sym, A_mu0_sym, EK, mu0K)

print("U_E_sym =")
show(U_E_sym)

print("U_mu0_sym =")
show(U_mu0_sym)

U_E_sym =


[                                                                                                                                     0                                                                                                                                      1                                                                                                                                      0]
[(-48*mu0*E + 48*E^2 - 96*mu0)/(-64*mu0*E^3 + 64*E^4 + 256*mu0^2*E - 128*mu0*E^2 - 192*E^3 - 512*mu0^2 + 768*mu0*E + 128*E^2 - 512*mu0)                                                                                                                                      0                                              (-64*mu0^2*E + 112*mu0*E^2 - 32*E^3 - 64*mu0^2 - 64*mu0*E + 16*E^2 + 64*mu0)/(-12*E + 24)]
[                           (6*mu0 - 3*E)/(-4*mu0*E^3 + 4*E^4 + 16*mu0^2*E - 8*mu0*E^2 - 12*E^3 - 32*mu0^2 + 48*mu0*E + 8*E^2 - 32*mu0)                                                                                                                                      0                                                        (4*mu0*E^2 - 12*E^3 + 48*mu0^2 - 48*mu0*E + 52*E^2 - 16*mu0 - 32*E)/(-6*E + 12)]

U_mu0_sym =


[                                                                                                          0                                                                                                           0                                                                                                           1]
[(6*mu0 - 3*E)/(-4*mu0*E^3 + 4*E^4 + 16*mu0^2*E - 8*mu0*E^2 - 12*E^3 - 32*mu0^2 + 48*mu0*E + 8*E^2 - 32*mu0)                                                                                                           0                             (4*mu0*E^2 - 12*E^3 + 48*mu0^2 - 48*mu0*E + 52*E^2 - 16*mu0 - 32*E)/(-6*E + 12)]
[                                           (-3)/(-4*mu0*E^2 + 4*E^3 + 16*mu0^2 - 16*mu0*E - 4*E^2 + 16*mu0)                                                                                                           0                                                                         -4/3*E^2 + 32/3*mu0 - 16/3*E + 16/3]

In [22]:
# Paper variables are called Ep, mup here
Rp.<Ep,mup> = PolynomialRing(QQ)
Kp = Rp.fraction_field()

Tp = (Ep - mup)*(4*mup + (Ep - 1)^2)

UE_paper = matrix(Kp, [
    [0,
     (-Ep + 2*mup + 1)/(4*Tp),
     -(1 + Ep)/(4*Tp)],

    [1,
     -mup*(1 + Ep)/(2*Tp),
     (2*Ep^2 - 2*Ep + 4*mup)/(4*Tp)],

    [0,
     mup*(Ep - 2*mup - 1)/(2*Tp),
     mup*(1 + Ep)/(2*Tp)]
])

Um_paper = matrix(Kp, [
    [0,
     -(1 + Ep)/(4*Tp),
     (Ep^2 - Ep + 2*mup)/(4*mup*Tp)],

    [0,
     (Ep^2 - Ep + 2*mup)/(2*Tp),
     -1/(2*mup) + (-Ep^2 + 3*Ep - 4*mup)/(2*Tp)],

    [1,
     mup*(1 + Ep)/(2*Tp),
     -1/(2*mup) - (Ep^2 - Ep + 2*mup)/(2*Tp)]
])

In [23]:
# coefficient field
K = U_E_sym.base_ring()
mu0K, mu1K, mu2K, EK = K.gens()

def old_to_new(expr):
    return K(expr.subs({Ep: 1 - EK, mup: -mu0K}))

UE_p_to_y = UE_paper.apply_map(old_to_new)
Um_p_to_y = Um_paper.apply_map(old_to_new)

Try to but them in the same gauge

In [24]:

G = diagonal_matrix(K, [K(1), K(-1), K(-1)])

UE_expected = -G * UE_p_to_y * G
Um_expected = -G * Um_p_to_y * G

In [25]:
def matrix_to_K(M, K):
    return matrix(K, M.nrows(), M.ncols(),
                  lambda i,j: K(M[i,j]))

U_E_sym_K     = matrix_to_K(U_E_sym, K)
U_mu0_sym_K   = matrix_to_K(U_mu0_sym, K)
UE_expected_K = matrix_to_K(UE_expected, K)
Um_expected_K = matrix_to_K(Um_expected, K)

def simp(x):
    if x == 0:
        return x
    try:
        return x.factor()
    except Exception:
        return x

check_UE = (U_E_sym_K - UE_expected_K).apply_map(simp)
check_Um = (U_mu0_sym_K - Um_expected_K).apply_map(simp)

print("Check U_E:")
show(check_UE)

print("Check U_mu:")
show(check_Um)

TypeError: unable to find a common ring for all elements

In [26]:
def toK(x):
    return K(str(x))

def matrix_to_K_str(M, K):
    return matrix(K, M.nrows(), M.ncols(),
                  lambda i,j: toK(M[i,j]))

In [27]:
def matrix_to_SR(M):
    return matrix(SR, M.nrows(), M.ncols(),
                  lambda i,j: SR(M[i,j]))

U_E_sym_SR     = matrix_to_SR(U_E_sym)
U_mu0_sym_SR   = matrix_to_SR(U_mu0_sym)
UE_expected_SR = matrix_to_SR(UE_expected)
Um_expected_SR = matrix_to_SR(Um_expected)

check_UE = (U_E_sym_SR - UE_expected_SR).apply_map(lambda x: x.simplify_rational())
check_Um = (U_mu0_sym_SR - Um_expected_SR).apply_map(lambda x: x.simplify_rational())

print("Check U_E:")
show(check_UE)

print("Check U_mu:")
show(check_Um)

Check U_E:


[                                                                                                                                                                                                                                                                           0                                                                                                                                                                     1/4*(4*E^3 - 4*E^2 - 2*(2*E^2 + 8*E - 7)*mu0 + 16*mu0^2 + E)/(E^3 - E^2 - (E^2 + 4*E - 4)*mu0 + 4*mu0^2)                                                                                                                                                                                                                      1/4*(E - 2)/(E^3 - E^2 - (E^2 + 4*E - 4)*mu0 + 4*mu0^2)]
[                                                                                                                   -1/4*(4*E^4 - 12*E^3 + 16*(E - 2)*mu0^2 + 5*E^2 - (4*E^3 + 8*E^2 - 51*E + 26)*mu0)/(E^4 - 3*E^3 + 4*(E - 2)*mu0^2 + 2*E^2 - (E^3 + 2*E^2 - 12*E + 8)*mu0)                                                                                                                                                                                                                  1/2*(E - 2)*mu0/(E^3 - E^2 - (E^2 + 4*E - 4)*mu0 + 4*mu0^2) 1/6*(16*E^6 - 24*E^5 + 128*(E + 1)*mu0^4 + 8*E^4 - 32*(E^3 + 12*E^2 - 4*E)*mu0^3 - 3*E^3 + 8*(11*E^4 + 32*E^3 - 48*E^2 + 32*E - 16)*mu0^2 + 9*E^2 - 2*(36*E^5 - 16*E^4 - 16*E^3 - 3*E + 6)*mu0 - 6*E)/(E^4 - 3*E^3 + 4*(E - 2)*mu0^2 + 2*E^2 - (E^3 + 2*E^2 - 12*E + 8)*mu0)]
[                                                                                                                                                                                     -3/4*(E - 2*mu0)/(E^4 - 3*E^3 + 4*(E - 2)*mu0^2 + 2*E^2 - (E^3 + 2*E^2 - 12*E + 8)*mu0)                                                                                                                                                                                                           -1/2*(E*mu0 - 2*mu0^2)/(E^3 - E^2 - (E^2 + 4*E - 4)*mu0 + 4*mu0^2)                 1/6*(12*E^6 - 64*E^5 + 84*E^4 + 32*(E^2 + 12*E - 4)*mu0^3 - 192*mu0^4 - 32*E^3 + 4*(E^4 - 8*E^3 - 96*E^2 + 64*E + 16)*mu0^2 - (16*E^5 - 56*E^4 - 192*E^3 + 355*E^2 - 140*E + 12)*mu0)/(E^4 - 3*E^3 + 4*(E - 2)*mu0^2 + 2*E^2 - (E^3 + 2*E^2 - 12*E + 8)*mu0)]

Check U_mu:


[                                                                                                                                                                                                                                                                                                               0                                                                                                                                                                                                                                                          1/4*(E - 2)/(E^3 - E^2 - (E^2 + 4*E - 4)*mu0 + 4*mu0^2)                                                                                                                                                                               1/4*(4*(E^2 + 4*E - 4)*mu0^2 - 16*mu0^3 + E^2 - 2*(2*E^3 - 2*E^2 + 1)*mu0 - E)/((E^2 + 4*E - 4)*mu0^2 - 4*mu0^3 - (E^3 - E^2)*mu0)]
[                                                                                                                                                                                                                         -3/4*(E - 2*mu0)/(E^4 - 3*E^3 + 4*(E - 2)*mu0^2 + 2*E^2 - (E^3 + 2*E^2 - 12*E + 8)*mu0)                                                                                                                                                                                                                                               -1/2*(E^2 - E - 2*mu0)/(E^3 - E^2 - (E^2 + 4*E - 4)*mu0 + 4*mu0^2) 1/6*(32*(E^2 + 12*E - 4)*mu0^4 - 192*mu0^5 + 3*E^4 + 4*(E^4 - 8*E^3 - 96*E^2 + 64*E + 16)*mu0^3 - 9*E^3 - 8*(2*E^5 - 7*E^4 - 24*E^3 + 44*E^2 - 16*E)*mu0^2 + 6*E^2 + (12*E^6 - 64*E^5 + 84*E^4 - 32*E^3 - 9*E^2 + 24*E - 12)*mu0)/(4*(E - 2)*mu0^3 - (E^3 + 2*E^2 - 12*E + 8)*mu0^2 + (E^4 - 3*E^3 + 2*E^2)*mu0)]
[                                                                                                                                                                                                         -1/4*(4*E^3 - 4*E^2 - 4*(E^2 + 4*E - 4)*mu0 + 16*mu0^2 + 3)/(E^3 - E^2 - (E^2 + 4*E - 4)*mu0 + 4*mu0^2)                                                                                                                                                                                                                                                     -1/2*(E - 2)*mu0/(E^3 - E^2 - (E^2 + 4*E - 4)*mu0 + 4*mu0^2)                                                                                                      1/6*(96*(E^2 + 4*E - 4)*mu0^3 - 256*mu0^4 - 3*E^3 - 2*(4*E^4 + 64*E^3 - 128*E + 67)*mu0^2 + 3*E^2 + (8*E^5 + 24*E^4 - 64*E^3 + 32*E^2 + 15*E - 12)*mu0)/((E^2 + 4*E - 4)*mu0^2 - 4*mu0^3 - (E^3 - E^2)*mu0)]

Check whether the two are gauge equivalent

In [28]:
# relation row in derivative basis
dE, cE = A_E_sym[0,0], A_E_sym[0,1]
dM, cM = A_mu0_sym[0,0], A_mu0_sym[0,1]

rel = vector(SR, [
    symbolic_expression(-(cM*dE - cE*dM)),
    symbolic_expression(cM),
    symbolic_expression(-cE)
])

def row_equiv_mod_relation(row):
    """
    Check whether row is proportional to rel.
    Returns the proportionality candidates.
    """
    row = vector(SR, [symbolic_expression(x).simplify_rational() for x in row])

    # Check all 2x2 minors row_i rel_j - row_j rel_i
    minors = []
    for i in range(3):
        for j in range(i+1,3):
            minors.append((row[i]*rel[j] - row[j]*rel[i]).simplify_rational())

    return minors

for name, M in [("check_UE", check_UE), ("check_Um", check_Um)]:
    print(name)
    for i in range(3):
        mins = row_equiv_mod_relation([M[i,0], M[i,1], M[i,2]])
        print("row", i, [m.simplify_rational() for m in mins])

check_UE
row 0 [1/8*(4*E^3 - 4*E^2 - 2*(2*E^2 + 8*E - 7)*mu0 + 16*mu0^2 + E)/(E^4 - 3*E^3 + 4*(E - 2)*mu0^2 + 2*E^2 - (E^3 + 2*E^2 - 12*E + 8)*mu0), 1/8/(E^3 - E^2 - (E^2 + 4*E - 4)*mu0 + 4*mu0^2), 1/2*(2*E^4 - 2*E^3 + 2*(2*E^2 + 12*E - 7)*mu0^2 - 16*mu0^3 + E^2 - 2*(3*E^3 + 2*E^2 - 3*E)*mu0 - 2*E + 2)/(E^4 - 3*E^3 + 4*(E - 2)*mu0^2 + 2*E^2 - (E^3 + 2*E^2 - 12*E + 8)*mu0)]
row 1 [1/4*(4*E^4 - 12*E^3 + 16*(E - 2)*mu0^2 + 5*E^2 - 4*(E^3 + 2*E^2 - 13*E + 7)*mu0)/(E^4 - 3*E^3 + 4*(E - 2)*mu0^2 + 2*E^2 - (E^3 + 2*E^2 - 12*E + 8)*mu0), 1/12*(16*E^6 - 36*E^5 + 128*(E + 1)*mu0^4 + 44*E^4 - 32*(E^3 + 12*E^2 - 7*E + 6)*mu0^3 - 18*E^3 + 2*(44*E^4 + 116*E^3 - 240*E^2 + 329*E - 142)*mu0^2 + 9*E^2 - (72*E^5 - 68*E^4 + 16*E^3 + 123*E^2 - 84*E + 12)*mu0 - 6*E)/(E^5 - 5*E^4 + 8*E^3 + 4*(E^2 - 4*E + 4)*mu0^2 - 4*E^2 - (E^4 - 16*E^2 + 32*E - 16)*mu0), 1/6*(16*E^6 - 24*E^5 + 128*(E + 1)*mu0^4 + 8*E^4 - 32*(E^3 + 12*E^2 - 4*E)*mu0^3 - 3*E^3 + 2*(44*E^4 + 128*E^3 - 192*E^2 + 125*E - 58)*mu0^2 + 9*E^2 - (72*

In [29]:
# Paper U matrices act on coefficient vectors.
# Our U_E_sym, U_mu0_sym act on the derivative basis vector.
# So maybe we should  compare against transpose of paper U...?

UE_expected = -G * UE_p_to_y.transpose() * G
Um_expected = -G * Um_p_to_y.transpose() * G

In [30]:
U_E_sym_SR     = matrix_to_SR(U_E_sym)
U_mu0_sym_SR   = matrix_to_SR(U_mu0_sym)
UE_expected_SR = matrix_to_SR(UE_expected)
Um_expected_SR = matrix_to_SR(Um_expected)

check_UE = (U_E_sym_SR - UE_expected_SR).apply_map(lambda x: x.simplify_rational())
check_Um = (U_mu0_sym_SR - Um_expected_SR).apply_map(lambda x: x.simplify_rational())

show(check_UE)
show(check_Um)

[                                                                                                                                     0                                                                                                                                      0                                                                                                                                      0]
[                              1/4*(4*E^2 - (5*E + 2)*mu0 - 2*E)/(E^4 - 3*E^3 + 4*(E - 2)*mu0^2 + 2*E^2 - (E^3 + 2*E^2 - 12*E + 8)*mu0)                                                                            1/2*(E - 2)*mu0/(E^3 - E^2 - (E^2 + 4*E - 4)*mu0 + 4*mu0^2) -1/2*(4*E^3 + 6*(E + 2)*mu0^2 - 2*E^2 - (13*E^2 - 6*E + 8)*mu0)/(E^4 - 3*E^3 + 4*(E - 2)*mu0^2 + 2*E^2 - (E^3 + 2*E^2 - 12*E + 8)*mu0)]
[                                    1/4*(E^2 - 7*E + 6*mu0 + 4)/(E^4 - 3*E^3 + 4*(E - 2)*mu0^2 + 2*E^2 - (E^3 + 2*E^2 - 12*E + 8)*mu0)                                                                     -1/2*(E^2 - E - 2*mu0)/(E^3 - E^2 - (E^2 + 4*E - 4)*mu0 + 4*mu0^2)          -1/2*(3*E^3 - 13*E^2 + 8*(E + 1)*mu0 - 12*mu0^2 + 8*E)/(E^4 - 3*E^3 + 4*(E - 2)*mu0^2 + 2*E^2 - (E^3 + 2*E^2 - 12*E + 8)*mu0)]

[                                                                                                                            0                                                                                                                             0                                                                                                                             0]
[                           1/4*(E^2 - 7*E + 6*mu0 + 4)/(E^4 - 3*E^3 + 4*(E - 2)*mu0^2 + 2*E^2 - (E^3 + 2*E^2 - 12*E + 8)*mu0)                                                            -1/2*(E^2 - E - 2*mu0)/(E^3 - E^2 - (E^2 + 4*E - 4)*mu0 + 4*mu0^2) -1/2*(3*E^3 - 13*E^2 + 8*(E + 1)*mu0 - 12*mu0^2 + 8*E)/(E^4 - 3*E^3 + 4*(E - 2)*mu0^2 + 2*E^2 - (E^3 + 2*E^2 - 12*E + 8)*mu0)]
[                                                      1/4*(E^2 - E + mu0)/((E^2 + 4*E - 4)*mu0^2 - 4*mu0^3 - (E^3 - E^2)*mu0)                                          -1/2*(E^3 - E^2 - (3*E - 2)*mu0)/((E^2 + 4*E - 4)*mu0^2 - 4*mu0^3 - (E^3 - E^2)*mu0)                       -1/2*(E^3 - E^2 + (2*E^2 + 3*E - 4)*mu0 - 14*mu0^2)/((E^2 + 4*E - 4)*mu0^2 - 4*mu0^3 - (E^3 - E^2)*mu0)]

In [31]:
Delta_E = U_E_sym_SR - UE_expected_SR
Delta_M = U_mu0_sym_SR - Um_expected_SR

In [32]:
(U_E_sym_SR).trace() - (UE_expected_SR).trace()
(U_E_sym_SR).det()   - (UE_expected_SR).det()

3/8*(3*E^3 - E^2*mu0 - 13*E^2 + 12*E*mu0 - 12*mu0^2 + 8*E + 4*mu0)*(E^2 - E*mu0 - 2*mu0)/(E^4 - E^3*mu0 - 3*E^3 - 2*E^2*mu0 + 4*E*mu0^2 + 2*E^2 + 12*E*mu0 - 8*mu0^2 - 8*mu0)^2 + 3/4*(2*E^3 - 7*E^2*mu0 + 4*E*mu0^2 - E^2 + 4*E*mu0 + 4*mu0^2 - 4*mu0)*(E - 2*mu0)/(E^4 - E^3*mu0 - 3*E^3 - 2*E^2*mu0 + 4*E*mu0^2 + 2*E^2 + 12*E*mu0 - 8*mu0^2 - 8*mu0)^2 - 1/8*(E*mu0 - 2*mu0)*(E - 2*mu0)/(E^3 - E^2*mu0 - E^2 - 4*E*mu0 + 4*mu0^2 + 4*mu0)^2 + 1/8*(E*mu0 - 2*mu0^2)*(E - 2)/(E^3 - E^2*mu0 - E^2 - 4*E*mu0 + 4*mu0^2 + 4*mu0)^2

In [33]:
# rescale derivative basis
S = diagonal_matrix([1, 1/cE, 1/cM])

U_E_rescaled  = S.inverse() * U_E_sym * S
U_mu_rescaled = S.inverse() * U_mu0_sym * S

In [34]:
U_E_rescaled_SR  = matrix_to_SR(U_E_rescaled)
U_mu_rescaled_SR = matrix_to_SR(U_mu_rescaled)

check_UE = (U_E_rescaled_SR - UE_expected_SR).apply_map(lambda x: x.simplify_rational())
check_Um = (U_mu_rescaled_SR - Um_expected_SR).apply_map(lambda x: x.simplify_rational())

show(check_UE)
show(check_Um)


[                                                                                                                                                                                   0                                                                                                                                                         -2*(E - mu0 - 1)/(E - 2*mu0)                                                                                                                                                                                    0]
[                      -1/4*(2*E^3 + 6*(E + 2)*mu0^2 + 4*E^2 - (7*E^2 + 14*E - 8)*mu0 - 4*E)/(E^5 - 5*E^4 + 8*E^3 + 4*(E^2 - 4*E + 4)*mu0^2 - 4*E^2 - (E^4 - 16*E^2 + 32*E - 16)*mu0)                                                                                                                          1/2*(E - 2)*mu0/(E^3 - E^2 - (E^2 + 4*E - 4)*mu0 + 4*mu0^2) -1/2*(4*E^4 - 16*(E + 1)*mu0^3 - 2*E^3 + 2*(17*E^2 + 4)*mu0^2 - (21*E^3 - 8*E^2 + 4*E)*mu0)/(E^5 - 5*E^4 + 8*E^3 + 4*(E^2 - 4*E + 4)*mu0^2 - 4*E^2 - (E^4 - 16*E^2 + 32*E - 16)*mu0)]
[                                                                                    1/4*(E^2 - E - 6*mu0 + 4)/(E^4 - 3*E^3 + 4*(E - 2)*mu0^2 + 2*E^2 - (E^3 + 2*E^2 - 12*E + 8)*mu0)                                                                                                                   -1/2*(E^2 - E - 2*mu0)/(E^3 - E^2 - (E^2 + 4*E - 4)*mu0 + 4*mu0^2)                                                        -1/2*(3*E^3 - 13*E^2 + 8*(E + 1)*mu0 - 12*mu0^2 + 8*E)/(E^4 - 3*E^3 + 4*(E - 2)*mu0^2 + 2*E^2 - (E^3 + 2*E^2 - 12*E + 8)*mu0)]

[                                                                                                                                                                                        0                                                                                                                                                                                         0                                                                                                                                                                                        -2]
[                                              1/4*(E^3 - 3*E^2 - 12*E*mu0 + 12*mu0^2 + 12*E - 8)/(E^5 - 5*E^4 + 8*E^3 + 4*(E^2 - 4*E + 4)*mu0^2 - 4*E^2 - (E^4 - 16*E^2 + 32*E - 16)*mu0)                                                                                                                        -1/2*(E^2 - E - 2*mu0)/(E^3 - E^2 - (E^2 + 4*E - 4)*mu0 + 4*mu0^2) -1/2*(3*E^4 - 13*E^3 + 2*(E^2 - 18*E - 4)*mu0^2 + 24*mu0^3 + 8*E^2 - 2*(3*E^3 - 16*E^2 + 4)*mu0)/(E^5 - 5*E^4 + 8*E^3 + 4*(E^2 - 4*E + 4)*mu0^2 - 4*E^2 - (E^4 - 16*E^2 + 32*E - 16)*mu0)]
[                                                                                                                1/4*(E^2 - E - 5*mu0)/((E^2 + 4*E - 4)*mu0^2 - 4*mu0^3 - (E^3 - E^2)*mu0)                                                                                                      -1/2*(E^3 - E^2 - (3*E - 2)*mu0)/((E^2 + 4*E - 4)*mu0^2 - 4*mu0^3 - (E^3 - E^2)*mu0)                                                                                   -1/2*(E^3 - E^2 + (2*E^2 + 3*E - 4)*mu0 - 14*mu0^2)/((E^2 + 4*E - 4)*mu0^2 - 4*mu0^3 - (E^3 - E^2)*mu0)]